In [1]:
import astropy.io.fits as fits
import astropy.units as u
from astropy.coordinates import SkyCoord
import numpy as np
import os
import pandas as pd
import glob

from create_figs import create_m0_map, mark_sources, plot_vector, plot_dotted_vector

import warnings
warnings.filterwarnings("ignore")

<Figure size 600x600 with 0 Axes>

<Figure size 600x600 with 0 Axes>

In [2]:
# helper function to extract channel indices from the data table
def getIdx(listOf):
    ranges = []
    
    for string in listOf:
        if pd.isna(string):  # Skip NaN values
            continue
        
        for part in string.split(', '):
            if '-' in part:  # Handle ranges
                start, end = map(int, part.split('-'))
                ranges.append(np.r_[start:end + 1])  # Use np.r_
            else:  # Handle single values
                ranges.append(int(part))  # Append as integer
    
    # Ensure `ranges` is not empty before passing to np.r_
    if not ranges:
        return np.r_[:]  # Returns an empty np.r_

    return np.r_[tuple(ranges)]  # Use `tuple(ranges)` to avoid errors

In [3]:
# read data
source_info = pd.read_csv("../data/output/source_info.csv")
source_info['Main'] = source_info['Main'].apply(lambda x: str(x).casefold())
source_info.set_index('Main', inplace=True)

df = pd.read_csv('../data/output/outflow_data.csv')
df

,group,field,source_a,source_a_ra,source_a_dec,source_b,source_b_ra,source_b_dec,distance,outflow_source,red_channels,blue_channels,red_outflow_PA,blue_outflow_PA,outflow_PA,binary_PA,delta_PA
0,orion,HOPS-12,B-A,83.787271,-5.931953,B-B,83.787312,-5.931911,388.6,B-A,92-97,103-107,158.050,156.195,157.1225,45.000000,67.877500
1,orion,HOPS-12,A,83.785962,-5.931844,NaN,NaN,NaN,388.6,A,92-97,NaN,156.980,NaN,156.9800,NaN,NaN
2,orion,HOPS-290,A,84.989071,-7.492528,B,84.988871,-7.492419,405.5,A,91-94,99-102,291.515,295.520,293.5175,298.442929,4.925429
3,orion,HOPS-290,A,84.989071,-7.492528,B,84.988871,-7.492419,405.5,B,91-95,NaN,227.590,230.185,228.8875,298.442929,69.555429
4,orion,HOPS-92,A-A,83.826404,-5.009150,A-B,83.826367,-5.009217,392.7,both,NaN,106-112,NaN,76.030,76.0300,209.357754,46.672246
5,orion,HOPS-92,B,83.826125,-5.009422,NaN,NaN,NaN,392.7,B,93-97,NaN,268.760,NaN,268.7600,NaN,NaN
6,orion,HOPS-288,A-A,84.983338,-7.507658,A-B,84.983358,-7.507694,405.5,both,85-91,NaN,56.370,NaN,56.3700,150.018361,86.351639
7,orion,HOPS-288,B,84.983471,-7.507767,NaN,NaN,NaN,405.5,B,NaN,102-108,NaN,211.770,211.7700,NaN,NaN
8,orion,HOPS-203,A,84.095271,-6.768508,B,84.095258,-6.768542,383.5,both,96-100,NaN,143.810,NaN,143.8100,200.556045,56.746045
9,orion,HOPS-203,C,84.095408,-6.769297,NaN,NaN,NaN,383.5,C,96-100,NaN,100.920,NaN,100.9200,NaN,NaN


In [4]:
# This loops through each source field with an angle measurement and creates
# a figure with the separation vector and outflow vector overlayed
for i, field in df.groupby('field').agg('first').reset_index().iterrows():

    # verify output path exists and
    # skip already existing files if you don't want to overwrite them
    output_folder = "../results/multi-outflow_plots-2"
    target_name = field['field']
    filename = f"{target_name}_outflow.pdf"
    output_path = os.path.join(output_folder, filename)

    # verify output path exists and
    if not os.path.exists(output_folder):
        os.mkdir(output_folder)

    # open image
    image_filename = (glob.glob(f'/Volumes/Alpha/Research/data/{target_name.casefold()}/*12co*.fits') + glob.glob(f'/Volumes/Alpha/Research/data/{target_name.casefold()}/*spw39*.fits'))[0]
    hdulist = fits.open(image_filename)
    hdu = hdulist[0]

    # set center and size of cutout
    center = SkyCoord(hdu.header['OBSRA'], hdu.header['OBSDEC'], unit=u.degree)
    size = np.array([39, 39]) * u.arcsecond
    distance = field['distance']

    # create figure
    channels = getIdx([field['red_channels'], field['blue_channels']])
    fig = create_m0_map(hdu, center, size, channels, 3, distance)
    fig.set_title(f"{target_name} 12CO M0")

    # add a marker at each source with legend
    target_info = source_info.loc[target_name.casefold()]
    mark_sources(fig, target_info)

    # plot binary separation angle
    center_origin = np.array([np.mean([field['source_a_ra'], field['source_b_ra']]), np.mean([field['source_a_dec'], field['source_b_dec']])])
    separation_angle_north = field['binary_PA']
    # draw separation vector in both directions
    plot_vector(fig, center_origin, separation_angle_north, color='white', length=0.005)
    plot_vector(fig, center_origin, separation_angle_north + 180, color='white', length=0.005)
    
    # plot each outflow vector
    for j, source in df[df['field'] == target_name].reset_index(drop=True).iterrows():
        # define vector origin at the outflow source
        if source['outflow_source'] == 'both':
            outflow_origin = center_origin
        elif source['outflow_source'] == source['source_a']:
            outflow_origin = np.array([source['source_a_ra'], source['source_a_dec']])
        else:
            outflow_origin = np.array([source['source_b_ra'], source['source_b_dec']])

        # plot outflow vector
        outflow_angle_north = source['outflow_PA']
        plot_vector(fig, outflow_origin, outflow_angle_north, color='red', length=0.005)

        # plot component outflow vectors
        if ~pd.isna(source['red_outflow_PA']):
            plot_dotted_vector(fig, outflow_origin, 180 + source['red_outflow_PA'], color='white', length=0.005)
        if ~pd.isna(source['blue_outflow_PA']):
            plot_dotted_vector(fig, outflow_origin, source['blue_outflow_PA'], color='white', length=0.005)
        
        # calculate delta_PA
        angle = np.abs(outflow_angle_north - separation_angle_north) % 180
        angle = np.min([angle, 180 - angle])
            
        # display angle between outflow and separation in top left corner
        source_label = source['outflow_source']
        if source_label == 'both':
            source_label = source['source_a']+'+'+source['source_b']
        fig.ax.text(30,fig.ax.get_xlim()[1]-50-40*(j), f"{source_label} : {np.abs(angle):.2f}°")

    fig.savefig(output_path)

INFO: Auto-setting vmin to  0.000e+00 [aplpy.core]
INFO: Auto-setting vmax to  2.852e-01 [aplpy.core]
INFO: Auto-setting resolution to 199.101 dpi [aplpy.core]
INFO: Auto-setting vmin to  0.000e+00 [aplpy.core]
INFO: Auto-setting vmax to  1.512e-01 [aplpy.core]
INFO: Auto-setting resolution to 282.697 dpi [aplpy.core]
INFO: Auto-setting vmin to  0.000e+00 [aplpy.core]
INFO: Auto-setting vmax to  2.710e-01 [aplpy.core]
INFO: Auto-setting resolution to 300 dpi [aplpy.core]
INFO: Auto-setting vmin to  0.000e+00 [aplpy.core]
INFO: Auto-setting vmax to  9.319e-03 [aplpy.core]
INFO: Auto-setting resolution to 282.697 dpi [aplpy.core]
INFO: Auto-setting vmin to  0.000e+00 [aplpy.core]
INFO: Auto-setting vmax to  2.025e-01 [aplpy.core]
INFO: Auto-setting resolution to 300 dpi [aplpy.core]
INFO: Auto-setting vmin to  0.000e+00 [aplpy.core]
INFO: Auto-setting vmax to  1.667e-02 [aplpy.core]
INFO: Auto-setting resolution to 292.135 dpi [aplpy.core]
INFO: Auto-setting vmin to  0.000e+00 [aplpy.cor

In [5]:
df[df['field'] == target_name]

,group,field,source_a,source_a_ra,source_a_dec,source_b,source_b_ra,source_b_dec,distance,outflow_source,red_channels,blue_channels,red_outflow_PA,blue_outflow_PA,outflow_PA,binary_PA,delta_PA
32,perseus,Per-emb-5,A,52.837258,30.758386,B,52.837258,30.758386,300.0,both,141-148,164-173,297.655,298.945,298.3,90.0,28.3


In [6]:
source_info

,Source,RA,Dec,Dis,LBol,TBol,Class,SigmaYSO,group
Main,,,,,,,,,
hh270vla1,HH270VLA1-A,87.894113,2.946114,430.1,7.34,32.0,0,2.2,orion
hh270vla1,HH270VLA1-B,87.894167,2.946078,430.1,7.34,32.0,0,2.2,orion
hops-12,HOPS-12-A,83.785962,-5.931844,388.6,6.26,42.0,0,49.8,orion
hops-12,HOPS-12-B-A,83.787271,-5.931953,388.6,6.26,42.0,0,49.8,orion
hops-12,HOPS-12-B-B,83.787312,-5.931911,388.6,6.26,42.0,0,49.8,orion
...,...,...,...,...,...,...,...,...,...
per-emb-36,Per-emb-36-B,52.239042,31.237808,300.0,NaN,NaN,NaN,NaN,perseus
per-emb-44,Per-emb-44-A,52.265667,31.267725,300.0,NaN,NaN,NaN,NaN,perseus
per-emb-44,Per-emb-44-B,52.265583,31.267722,300.0,NaN,NaN,NaN,NaN,perseus
